In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 08:53:14.209572: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 08:53:15.125795: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
        "num_of_workers": 1,
        "ips": ['172.190.116.144'],
        "ports": [50151]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 1,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 08:53:16,872 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 08:53:16,877 [DEBUG] [Rain] Rain is initialized
2023-07-05 08:53:16,886 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 08:53:16,892 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 08:53:16,896 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 08:53:16,901 [DEBUG] [LazyProvisioner] Provisioner is initialized
2023-07-05 08:53:16,907 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 08:53:16,912 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 08:53:16,919 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 08:53:16,963 [INFO] [Provisioner] provisioner is serving
2023-07-05 08:53:16,969 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 08:53:16,975 [INFO] [Coordinator] coordinator is serving
2023-07-05 08:53:16,978 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 08:53:16,992 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 08:53:16,995 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 08:53:16,998 [DEBUG] [LazyProvisioner] Creating 1 workers
2023-07-05 08:53:17,000 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 08:53:17,003 [INFO] [Worker_50151] Worker is running on port: 50151


2023-07-05 08:53:17,006 [DEBUG] [Provisioner] [Created workers]
IPs : ['172.190.116.144'], ports: [50151], statuses: [1], IDs : [1]
2023-07-05 08:53:17,012 [DEBUG] [DividerAmbassador] divider ambassador is serving
2023-07-05 08:53:17,016 [DEBUG] [DividerProxy] Training Started
2023-07-05 08:53:17,192 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-05 08:53:17,195 [DEBUG] [Provisioner] Received '' from the coordinator to send status
2023-07-05 08:53:17,196 [DEBUG] [Provisioner] Workers
IPs : ['172.190.116.144'], ports: [50151], statuses: [1], IDs : [1]
2023-07-05 08:53:17,198 [DEBUG] [DividerAmbassador] divider received: information from coordinator
2023-07-05 08:53:17,203 [DEBUG] [DeepLearning] Starting iteration 1/1
2023-07-05 08:53:17,210 [DEBUG] [DividerAmbassador] 172.190.116.144:50151
2023-07-05 08:54:31,490 [DEBUG] [DividerAmbassador] Error sending the data to the worker: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 6ms/step - loss: 2.2892 - accuracy: 0.1194

Test accuracy: 11.9%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 08:54:32,890 [INFO] [Provisioner] provisioner is serving
2023-07-05 08:54:32,891 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 08:54:32,892 [INFO] [Coordinator] coordinator is serving
2023-07-05 08:54:32,894 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 08:54:32,901 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 08:54:32,907 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 08:54:32,912 [DEBUG] [LazyProvisioner] Creating 1 workers
2023-07-05 08:54:32,917 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 08:54:32,925 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 08:54:32,925 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 08:54:32,933 [ERROR] [LazyProvisioner] Error creating the workers: 'LazyProvisioner' object has no attribute 'workers'


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 5ms/step - loss: 2.2207 - accuracy: 0.1841

Test accuracy: 18.4%
